In [ ]:
# --- Run provenance: start of run -------------------------------------
# Records when this notebook run started and where it ran. The last cell
# of the notebook records the stop time, the notebook's settings, software
# versions and checksums. Keep this cell first and run it before the
# analysis cells.
import os
import time

_prov_start = time.time()
print("provenance start:", time.strftime("%Y-%m-%dT%H:%M:%S"))
print("working directory:", os.getcwd())

In [1]:
import os
import mdtraj as md
import numpy as np
import parmed as pmd

In [2]:
n_trajs = 50
n_frames = 20_000
p = pmd.load_file('../template/setup/top.psf')
n_residues = p.topology.getNumResidues()

# allocate with NaNs so missing trajectories stay obvious
all_sasa = np.full((n_trajs, n_frames, n_residues), np.nan, dtype=np.float32)

In [ ]:

for i in range(n_trajs):
    print(f"Calculating SASA for trajectory {i}")

    dcd_file = f'../{i}/aa_{i}.dcd'
    psf_file = f'../{i}/aa_{i}.psf'

    if not (os.path.exists(dcd_file) and os.path.exists(psf_file)):
        raise FileNotFoundError(f"Missing files for traj {i}: {dcd_file} or {psf_file}")

    # load trajectory
    traj = md.load(dcd_file, top=psf_file)

    # enforce frame count check
    if traj.n_frames != n_frames:
        raise ValueError(
            f"Trajectory {i} has {traj.n_frames} frames, expected {n_frames}. "
            "Stopping execution."
        )
    
    # calculate SASA per residue (frames × residues)
    sasa = md.shrake_rupley(traj, mode='residue')

    # write to array
    all_sasa[i] = sasa

# save to disk
np.save('SASA.npy', all_sasa)
print("Finished. SASA.npy saved.")

Calculating SASA for trajectory 0


In [ ]:
# --- Run provenance: end of run ---------------------------------------
# Prints the stop time, the values of the notebook's own variables (its
# settings), interpreter and package versions, and sha256 checksums of the
# files whose paths were written in the executed cells and of the files
# this run wrote. Keep this cell last.
import os
import re
import time
import types
import hashlib
import platform
from importlib.metadata import distributions


def _prov_sha256(path):
    _h = hashlib.sha256()
    with open(path, "rb") as _fh:
        for _chunk in iter(lambda: _fh.read(1 << 20), b""):
            _h.update(_chunk)
    return _h.hexdigest()


_prov_t0 = globals().get("_prov_start", time.time())
print(
    "provenance end:",
    time.strftime("%Y-%m-%dT%H:%M:%S"),
    f"(wall {time.time() - _prov_t0:.1f} s)",
)
try:
    print("python:", platform.python_version())
    _pkgs = sorted(
        f"{_d.metadata['Name']}=={_d.version}"
        for _d in distributions()
        if _d.metadata and _d.metadata.get("Name")
    )
    print("packages:", "; ".join(_pkgs))
except Exception as _exc:
    print("packages: unavailable", _exc)

_skip = {"In", "Out", "get_ipython", "exit", "quit", "open"}
for _k, _v in sorted(globals().items()):
    try:
        if _k.startswith("_") or _k in _skip:
            continue
        if isinstance(_v, (types.ModuleType, types.FunctionType, type)):
            continue
        _s = repr(_v)
        print(f"setting {_k} = {_s[:1000]}"
              + (" ...[truncated]" if len(_s) > 1000 else ""))
        if isinstance(_v, (str, os.PathLike)) and os.path.isfile(os.fspath(_v)):
            print(f"sha256 {_k} ({os.fspath(_v)}): {_prov_sha256(os.fspath(_v))}")
    except Exception:
        pass

try:
    _seen = set()
    for _src in (In if "In" in globals() else []):
        for _m in re.findall(
            r"['\"]([\w./\\-]+\.(?:pkl|npy|csv|tsv|dcd|psf|dat|xml|top|prm|txt|pdf|png|html))['\"]",
            _src,
        ):
            if _m in _seen:
                continue
            _seen.add(_m)
            try:
                if os.path.isfile(_m):
                    print(f"sha256 {_m}: {_prov_sha256(_m)}")
            except Exception:
                pass
except Exception:
    pass

try:
    for _n in sorted(os.listdir(".")):
        if _n.endswith(".ipynb"):
            continue
        _p = os.path.join(".", _n)
        if os.path.isfile(_p) and os.path.getmtime(_p) >= _prov_t0:
            print(f"sha256 output {_n}: {_prov_sha256(_p)}")
except Exception:
    pass